# Pears No. 4

### Importing packages

In [23]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

### Global params

In [24]:
np.random.seed(42)
n_obs = 1000

transition_matrix = np.array([
    [0.95, 0.05],
    [0.15, 0.85]
])

phi_0, sigma_0 = 0.4, 1.0
phi_1, sigma_1 = 0.9, 4.0

### Data

In [25]:
states = np.zeros(n_obs, dtype=int)

for t in range(1, n_obs):
    states[t] = np.random.choice([0, 1], p=transition_matrix[states[t-1]])

spread = np.zeros(n_obs)

for t in range(1, n_obs):
    if states[t] == 0:
        spread[t] = phi_0 * spread[t-1] + np.random.normal(0, sigma_0)
    else:
        spread[t] = phi_1 * spread[t-1] + np.random.normal(0, sigma_1)


### Fit

#### AR(1)

In [26]:
spread_series = pd.Series(spread, name="Spread")
ar_model = sm.tsa.SARIMAX(spread_series, order=(1, 0, 0), trend='c').fit(disp=False)

print(f"AIC: {ar_model.aic:.2f} | BIC: {ar_model.bic:.2f}")
print(f"Estimated Phi (mean reversion): {ar_model.arparams[0]:.4f}")
print(f"Estimated Variance: {ar_model.params['sigma2']:.3f}\n")

AIC: 4409.71 | BIC: 4424.44
Estimated Phi (mean reversion): 0.7734
Estimated Variance: 4.782



#### MS-AR(1)

In [27]:
ms_model = sm.tsa.MarkovAutoregression(
    spread_series, 
    k_regimes=2,
    order=1,
    switching_variance=True,
    trend='c'
).fit()

print(f"AIC: {ms_model.aic:.2f} | BIC: {ms_model.bic:.2f}")
print("\n")

print("regime 0")
print(f"Mean Reversion (Phi): {ms_model.params['ar.L1[0]']:.4f}")
print(f"Variance (Sigma^2):   {ms_model.params['sigma2[0]']:.4f}")
print("\n")

print("regime 1")
print(f"Mean Reversion (Phi): {ms_model.params['ar.L1[1]']:.4f}")
print(f"Variance (Sigma^2):   {ms_model.params['sigma2[1]']:.4f}")
print("\n")

print("transition matrix")
print(f"P(Stay in Reg 0): {ms_model.params['p[0->0]']:.4f}")
p_1_to_1 = 1 - ms_model.params['p[1->0]']
print(f"P(Stay in Reg 1): {p_1_to_1:.4f}")
print("\n")


AIC: 3735.90 | BIC: 3775.15


regime 0
Mean Reversion (Phi): 0.3340
Variance (Sigma^2):   0.9240


regime 1
Mean Reversion (Phi): 0.8615
Variance (Sigma^2):   13.8867


transition matrix
P(Stay in Reg 0): 0.9600
P(Stay in Reg 1): 0.8911


